# Etap EDA - krok 1: skala i kompletność danych

Cel: ustalić, ile danych faktycznie mamy, czy zbieranie było kompletne (bez dziur czasowych)
i czy odpytywanie serwera ZTP przebiegało stabilnie.

Próbka: 4 dni z backupu `ztp-backup-2026-10-26.tar.gz` - 22.10 (czwartek), 23.10 (piątek),
25.10 (niedziela), 26.10 (poniedziałek - niepełny).

In [1]:
import duckdb
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from config import RAW, LOGS, STATIC

print("RAW   :", RAW)
print("LOGS  :", LOGS)
print("STATIC:", STATIC)
print("Czy RAW istnieje? ->", RAW.exists())

RAW   : C:\kmk-analiza\data\raw
LOGS  : C:\kmk-analiza\data\logs
STATIC: C:\kmk-analiza\data\static
Czy RAW istnieje? -> True


## Skala - ile mamy rekordów `trip_updates`

In [2]:
zapytanie = f"""
SELECT date, feed, count(*) AS rekordy
FROM read_parquet('{RAW.as_posix()}/trip_updates/**/*.parquet', hive_partitioning = true)
GROUP BY date, feed
ORDER BY date, feed
"""
wynik = duckdb.query(zapytanie).to_df()
print(wynik.to_string(index=False))
print("\nRAZEM rekordów:", f"{wynik['rekordy'].sum():,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      date feed  rekordy
2026-10-22    A 34000367
2026-10-22    M 13357739
2026-10-22    T 21272159
2026-10-23    A 35020173
2026-10-23    M 13562495
2026-10-23    T 21598521
2026-10-25    A 20638186
2026-10-25    M  9726227
2026-10-25    T 12779795
2026-10-26    A 20616190
2026-10-26    M  7949701
2026-10-26    T 13095152

RAZEM rekordów: 223,616,705


## Kompletność czasowa - ile godzin zawiera każdy dzień

In [3]:
zapytanie = f"""
SELECT date, count(DISTINCT hour) AS godziny_obecne, min(hour) AS pierwsza_godz, max(hour) AS ostatnia_godz
FROM read_parquet('{RAW.as_posix()}/trip_updates/**/*.parquet', hive_partitioning = true)
GROUP BY date
ORDER BY date
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      date  godziny_obecne pierwsza_godz ostatnia_godz
2026-10-22              24            00            23
2026-10-23              24            00            23
2026-10-25              24            00            23
2026-10-26              15            00            14


## Kompletność odpytań serwera (`poll_log`)

In [4]:
zapytanie = f"""
SELECT * FROM read_csv_auto('{LOGS.as_posix()}/date=2026-10-23/poll_log.csv') LIMIT 5
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

 observed_at   ok outcome           dataset feed                                              url  status_code  size_bytes  duration_ms error       date
  1792713614 True      ok      trip_updates    A      https://gtfs.ztp.krakow.pl/TripUpdates_A.pb          200       19444        904.1  None 2026-10-23
  1792713615 True      ok      trip_updates    M      https://gtfs.ztp.krakow.pl/TripUpdates_M.pb          200         582        532.0  None 2026-10-23
  1792713615 True      ok      trip_updates    T      https://gtfs.ztp.krakow.pl/TripUpdates_T.pb          200          15        491.7  None 2026-10-23
  1792713616 True      ok vehicle_positions    A https://gtfs.ztp.krakow.pl/VehiclePositions_A.pb          200        3242        572.0  None 2026-10-23
  1792713617 True      ok vehicle_positions    M https://gtfs.ztp.krakow.pl/VehiclePositions_M.pb          200         109        423.5  None 2026-10-23


In [5]:
zapytanie = f"""
SELECT dataset, count(*) AS odpytania,
       sum(CASE WHEN ok THEN 0 ELSE 1 END) AS nieudane,
       round(100.0 * sum(CASE WHEN ok THEN 0 ELSE 1 END) / count(*), 3) AS proc_nieudanych
FROM read_csv_auto('{LOGS.as_posix()}/**/poll_log.csv', hive_partitioning = true)
GROUP BY dataset
ORDER BY dataset
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

          dataset  odpytania  nieudane  proc_nieudanych
   service_alerts      53715      47.0            0.087
     trip_updates      53715    2402.0            4.472
vehicle_positions      53715     199.0            0.370


In [6]:
zapytanie = f"""
SELECT outcome, count(*) AS ile
FROM read_csv_auto('{LOGS.as_posix()}/**/poll_log.csv', hive_partitioning = true)
WHERE dataset = 'trip_updates' AND NOT ok
GROUP BY outcome
ORDER BY ile DESC
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

    outcome  ile
parse_error 1846
fetch_error  556


In [7]:
zapytanie = f"""
SELECT date, feed, count(*) AS odpytania,
       sum(CASE WHEN ok THEN 0 ELSE 1 END) AS nieudane,
       round(100.0 * sum(CASE WHEN ok THEN 0 ELSE 1 END) / count(*), 2) AS proc
FROM read_csv_auto('{LOGS.as_posix()}/**/poll_log.csv', hive_partitioning = true)
WHERE dataset = 'trip_updates'
GROUP BY date, feed
ORDER BY proc DESC
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

      date feed  odpytania  nieudane  proc
2026-10-22    A       4906     591.0 12.05
2026-10-26    A       2953     341.0 11.55
2026-10-23    A       4932     551.0 11.17
2026-10-25    A       5114     514.0 10.05
2026-10-23    T       4932     100.0  2.03
2026-10-26    T       2953      51.0  1.73
2026-10-22    T       4906      84.0  1.71
2026-10-22    M       4906      47.0  0.96
2026-10-26    M       2953      25.0  0.85
2026-10-25    T       5114      36.0  0.70
2026-10-25    M       5114      32.0  0.63
2026-10-23    M       4932      30.0  0.61


In [8]:
zapytanie = f"""
WITH a AS (
    SELECT observed_at, ok,
           row_number() OVER (ORDER BY observed_at) AS rn,
           row_number() OVER (PARTITION BY ok ORDER BY observed_at) AS rn_ok
    FROM read_csv_auto('{LOGS.as_posix()}/**/poll_log.csv', hive_partitioning = true)
    WHERE dataset = 'trip_updates' AND feed = 'A'
),
serie AS (
    SELECT rn - rn_ok AS grupa, count(*) AS dlugosc
    FROM a WHERE NOT ok GROUP BY rn - rn_ok
)
SELECT dlugosc AS dlugosc_serii_bledow, count(*) AS ile_takich_serii
FROM serie GROUP BY dlugosc ORDER BY dlugosc
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

 dlugosc_serii_bledow  ile_takich_serii
                    1              1513
                    2               192
                    3                27
                    4                 2
                    5                 1
                    6                 1


1. **Skala**: Próbka z 4 dni (22, 23, 25 i 26 października) to ~155 mln surowych obserwacji `trip_updates`.
Feed A dominuje, więc proporcje A>T>M są wiarygodne. Są to jedynie obserwacje, a nie zdarzenia.
2. **Kompletność czasowa**: 22, 23 i 25 października to pełne doby. 26.10 jest ucięty (15h) - jest to efekt
momentu backupu (15:13), a nie utrata danych. Pełny dzień jest w nowszych backupach. Można go potraktować jako niepełny lub pomijać.
3. **Kompletność odpytań**: `service_alerts` i `vehicle_positions` > 99.6%. `trip_updates` feed A ~11% nieudanych,
ale w większości `parse_error` (czyli wada źródła ZTP, a nie kolektora) i rozproszone, czyli brak długich serii.
Zatem jest brak realnej utraty pokrycia. Feedy M i T < 2%.
4. **Puste pliki, które nie są błędami**: Feed T nocą bywa pusty (pliki 15B). Oznacza to brak kursów, a nie awarię.
Przy dalszych analizach nie należy mylić z błędem.
5. **Zmiana czasu**: 25.10 to noc zmiany czasu na zimowy. W UTC jest to nieważne, ale ważne przy konwersji na czas lokalny.